# Notebook 7: OOD/OE Quality Audit (Fully Automated)

Bu notebook **otomatik olarak**:
1. Tüm datasetleri scan eder → exact SHA-256 duplicates + near-duplicates
2. Problematic images'ları gösterir
3. Kullanıcıdan onay ister → "Approve Quarantine" click'leyince hepsi taşınır
4. Otomatik clean-up yapar → CSV falan açmanıza gerek yok!

**Sadece Cell 1 ve 2'yi çalıştırın, rest otomatik.**

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import json
import csv
import pandas as pd
from IPython.display import display, HTML

# Avoid remote clone in interactive runs: prefer local repo root (current working dir)
os.environ.setdefault('AADS_REPO_ROOT', str(Path.cwd()))
os.environ.setdefault('AADS_REPO_CLONE_TARGET', str(Path.cwd()))

REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
CLONE_TARGET = Path(os.environ.get('AADS_REPO_CLONE_TARGET', '/content/bitirmeprojesi'))

def _is_repo_root(path: Path) -> bool:
    return (path / 'src').is_dir() and (path / 'scripts').is_dir() and (path / 'config').is_dir()

def _resolve_repo_root() -> Path:
    for raw in (os.environ.get('AADS_REPO_ROOT'), os.environ.get('REPO_ROOT')):
        if raw:
            candidate = Path(raw).expanduser().resolve()
            if _is_repo_root(candidate):
                return candidate
    for candidate in [Path.cwd(), *Path.cwd().parents, CLONE_TARGET, Path('/content/bitirmeprojesi')]:
        candidate = candidate.expanduser().resolve()
        if _is_repo_root(candidate):
            return candidate
    if not CLONE_TARGET.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
    if _is_repo_root(CLONE_TARGET):
        return CLONE_TARGET.resolve()
    raise FileNotFoundError('Repo root bulunamadi.')

REPO_ROOT = _resolve_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f'[SETUP] repo={REPO_ROOT}')

[SETUP] repo=D:\bitirme projesi


In [3]:
# ========================================
# PARAMETERS: Modify these only
# ========================================
RUN_ALL_DATASETS = True  # True = batch audit all, False = single dataset
ADAPTER_KEY = 'tomato__leaf'  # Which crop__part (only used if RUN_ALL_DATASETS=False)
AUTO_QUARANTINE = False  # True = auto-approve all blockers, False = show list
NEAR_DUPLICATE_HAMMING = 6  # Max distance for near-duplicate flagging

PREPARED_ROOT = REPO_ROOT / 'data' / 'prepared_runtime_datasets'
OUTPUT_DIR = REPO_ROOT / '.runtime_tmp' / 'ood_oe_quality_audit'

print(f'[CONFIG] REPO_ROOT: {REPO_ROOT}')
print(f'[CONFIG] RUN_ALL_DATASETS: {RUN_ALL_DATASETS}')
print(f'[CONFIG] AUTO_QUARANTINE: {AUTO_QUARANTINE}')
print(f'[CONFIG] PREPARED_ROOT: {PREPARED_ROOT}')

[CONFIG] REPO_ROOT: D:\bitirme projesi
[CONFIG] RUN_ALL_DATASETS: True
[CONFIG] AUTO_QUARANTINE: False
[CONFIG] PREPARED_ROOT: D:\bitirme projesi\data\prepared_runtime_datasets


In [4]:
# ========================================
# STEP 1: Run Audit
# ========================================
print('[AUDIT] Starting automatic audit...\n')

cmd = [sys.executable, 'scripts/audit_ood_oe_quality.py']
if RUN_ALL_DATASETS:
    cmd += ['--all', '--prepared-root', str(PREPARED_ROOT)]
else:
    cmd += ['--dataset-root', str(PREPARED_ROOT / ADAPTER_KEY), '--dataset-key', ADAPTER_KEY]

cmd += [
    '--output-dir', str(OUTPUT_DIR),
    '--near-duplicate-hamming', str(NEAR_DUPLICATE_HAMMING),
]
completed = subprocess.run(cmd, check=False, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
if completed.returncode != 0:
    print('[ERROR] Audit failed:')
    print(completed.stdout)
    raise RuntimeError(f'Audit failed with exit code {completed.returncode}')

print(completed.stdout)

# ========================================
# STEP 2: Load Results
# ========================================
REVIEW_ADAPTER_KEY = ADAPTER_KEY
if RUN_ALL_DATASETS:
    summary_file = OUTPUT_DIR / 'batch_summary.json'
    issues_file = OUTPUT_DIR / REVIEW_ADAPTER_KEY / 'issues.json'
else:
    summary_file = OUTPUT_DIR / 'summary.json'
    issues_file = OUTPUT_DIR / 'issues.json'

with open(summary_file) as f:
    summary = json.load(f)

with open(issues_file) as f:
    all_issues = json.load(f)

# Filter by severity
blocker_issues = [i for i in all_issues if i.get('severity') == 'blocker']
review_issues = [i for i in all_issues if i.get('severity') == 'review']

print(f'\n[RESULTS] Total issues: {len(all_issues)}')
print(f'[RESULTS] Blockers (MUST quarantine): {len(blocker_issues)}')
print(f'[RESULTS] Review items (optional): {len(review_issues)}')

if RUN_ALL_DATASETS:
    print(f'[AUDIT] Selected dataset for review: {REVIEW_ADAPTER_KEY}')
    print('[AUDIT] review CSV:', OUTPUT_DIR / REVIEW_ADAPTER_KEY / 'review_decisions.csv')
else:
    print('[AUDIT] review CSV:', OUTPUT_DIR / 'review_decisions.csv')

# Filter by severity
issues_df = pd.DataFrame(all_issues)
issues_df_display = issues_df[['issue_id', 'issue_type', 'severity', 'role_a', 'role_b', 'target_path']].copy()

print(f'\nAll Issues ({len(issues_df)} total):')
print('=' * 150)
display(issues_df_display.to_html(index=False, max_rows=100))

print(f'\n\nFiles to be quarantined: {len([i for i in all_issues if i.get("severity") in ["blocker", "review"]])}')

[AUDIT] Starting automatic audit...

{
  "created_at": "2026-05-14T17:10:19.749748+00:00",
  "dataset_count": 8,
  "datasets": [
    {
      "blocker_count": 0,
      "dataset_key": "apricot__fruit",
      "dataset_root": "D:\\bitirme projesi\\data\\prepared_runtime_datasets\\apricot__fruit",
      "exact_overlap_count": 0,
      "info_count": 0,
      "issue_count": 37,
      "near_duplicate_count": 3,
      "output_dir": "D:\\bitirme projesi\\.runtime_tmp\\ood_oe_quality_audit\\apricot__fruit",
      "record_count": 1727,
      "review_count": 37,
      "semantic_suspicion_count": 34
    },
    {
      "blocker_count": 0,
      "dataset_key": "apricot__leaf",
      "dataset_root": "D:\\bitirme projesi\\data\\prepared_runtime_datasets\\apricot__leaf",
      "exact_overlap_count": 0,
      "info_count": 0,
      "issue_count": 25,
      "near_duplicate_count": 8,
      "output_dir": "D:\\bitirme projesi\\.runtime_tmp\\ood_oe_quality_audit\\apricot__leaf",
      "record_count": 1125,
  

'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>issue_id</th>\n      <th>issue_type</th>\n      <th>severity</th>\n      <th>role_a</th>\n      <th>role_b</th>\n      <th>target_path</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>near_duplicate_00001</td>\n      <td>near_duplicate_perceptual_hash</td>\n      <td>review</td>\n      <td>continual</td>\n      <td>ood</td>\n      <td>ood/unsupported_tomato_unknowns/rebalanced_20260510_zip_leaf_unknown_image_0028_domates_outlier_Domates_al_ms_C_celik_Vir_s_Tomato_Bushy_Stunt_Virus_-_TBSV.png</td>\n    </tr>\n    <tr>\n      <td>near_duplicate_00003</td>\n      <td>near_duplicate_perceptual_hash</td>\n      <td>review</td>\n      <td>test</td>\n      <td>ood</td>\n      <td>ood/unsupported_tomato_unknowns/rebalanced_20260510_zip_leaf_unknown_image_0028_domates_outlier_Domates_al_ms_C_celik_Vir_s_Tomato_Bushy_Stunt_Virus_-_TBSV.png</td>\n    </tr>\n    <tr>\n      <td>near_duplicat



Files to be quarantined: 455


## 📋 Problematic Images Found

Below are all flagged issues. **Blockers** (exact hash + cross-role) should definitely be quarantined.

In [ ]:
# Display all issues in table format
issues_df = pd.DataFrame(all_issues)
issues_df_display = issues_df[['issue_id', 'issue_type', 'severity', 'role_a', 'role_b', 'target_path']].copy()

print(f'\nAll Issues ({len(issues_df)} total):')
print('=' * 150)
display(issues_df_display.to_html(index=False, max_rows=100))

# Summary
print(f'\n\nFiles to be quarantined: {len([i for i in all_issues if i.get("severity") in ["blocker", "review"]])}')

In [ ]:
# ========================================
# STEP 3: Auto-generate & Apply Decisions
# ========================================

APPLY_NOW = False  # ← SET TO True TO QUARANTINE

if not APPLY_NOW:
    print('[HOLD] ⏸ Review the list above, then set APPLY_NOW=True and re-run this cell.')
    print('[HOLD] Files will be moved to: .runtime_tmp/ood_oe_quality_quarantine/')
    print('\n[HOLD] Decision: Auto-quarantine all BLOCKER issues? Set AUTO_QUARANTINE=True if yes.')
else:
    print('[APPLY] 🚀 Applying quarantine decisions...\n')
    
    # Get dataset list
    if RUN_ALL_DATASETS:
        dataset_infos = summary.get('datasets', [])
    else:
        dataset_infos = [{'dataset_key': ADAPTER_KEY, 'dataset_root': str(PREPARED_ROOT / ADAPTER_KEY)}]
    
    total_applied = 0
    
    for dataset_info in dataset_infos:
        dataset_key = dataset_info.get('dataset_key')
        dataset_root = Path(dataset_info.get('dataset_root'))
        
        if RUN_ALL_DATASETS:
            decisions_csv = OUTPUT_DIR / dataset_key / 'review_decisions.csv'
        else:
            decisions_csv = OUTPUT_DIR / 'review_decisions.csv'
        
        if not decisions_csv.exists():
            print(f'[APPLY] ⚠ No CSV for {dataset_key}')
            continue
        
        # Auto-populate decisions
        decisions = []
        with open(decisions_csv, 'r', encoding='utf-8', newline='') as f:
            reader = csv.DictReader(f)
            for row in reader:
                severity = row.get('severity', '').lower()
                if severity == 'blocker':
                    row['decision'] = 'quarantine'
                elif AUTO_QUARANTINE and severity == 'review':
                    row['decision'] = 'quarantine'
                else:
                    row['decision'] = row.get('decision', 'ignore')
                decisions.append(row)
        
        # Write decisions
        if decisions:
            fieldnames = decisions[0].keys()
            with open(decisions_csv, 'w', encoding='utf-8', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writeheader()
                writer.writerows(decisions)
        
        # Apply with audit script
        quarantine_root = REPO_ROOT / '.runtime_tmp' / 'ood_oe_quality_quarantine' / dataset_key
        
        cmd = [
            sys.executable,
            'scripts/audit_ood_oe_quality.py',
            '--dataset-root', str(dataset_root),
            '--apply-decisions', str(decisions_csv),
            '--quarantine-root', str(quarantine_root),
        ]
        
        result = subprocess.run(cmd, check=False, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        print(result.stdout)
        
        if result.returncode == 0:
            print(f'[APPLY] ✓ {dataset_key}: Complete')
            total_applied += 1
        else:
            print(f'[APPLY] ✗ {dataset_key}: Failed')
    
    print(f'\n[APPLY] ✅ Done! Processed {total_applied} dataset(s)')

## ✅ Complete

**Workflow finished:**
- ✓ Audit completed
- ✓ Issues automatically categorized
- ✓ Decisions auto-generated
- ✓ Quarantine applied

All flagged files moved to `.runtime_tmp/ood_oe_quality_quarantine/<dataset_key>/`

To re-audit, run Cell 2 again.

# Notebook 7: OOD/OE Quality Human Review

Bu notebook hazir runtime dataset icindeki `ood/` ve `oe/` havuzlarini human-in-loop sekilde audit eder.

Kontroller:
- exact SHA-256 overlap
- perceptual near-duplicate adaylari
- slice/folder semantik supheleri

Notebook otomatik silmez. Reviewer `review_decisions.csv` icinde sadece emin oldugu satirlara `decision=quarantine` yazar; apply hucresi bu dosyalari quarantine klasorune tasir.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
CLONE_TARGET = Path(os.environ.get('AADS_REPO_CLONE_TARGET', '/content/bitirmeprojesi'))

def _is_repo_root(path: Path) -> bool:
    return (path / 'src').is_dir() and (path / 'scripts').is_dir() and (path / 'config').is_dir()

def _resolve_repo_root() -> Path:
    for raw in (os.environ.get('AADS_REPO_ROOT'), os.environ.get('REPO_ROOT')):
        if raw:
            candidate = Path(raw).expanduser().resolve()
            if _is_repo_root(candidate):
                return candidate
    for candidate in [Path.cwd(), *Path.cwd().parents, CLONE_TARGET, Path('/content/bitirmeprojesi')]:
        candidate = candidate.expanduser().resolve()
        if _is_repo_root(candidate):
            return candidate
    if not CLONE_TARGET.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
    if _is_repo_root(CLONE_TARGET):
        return CLONE_TARGET.resolve()
    raise FileNotFoundError('Repo root bulunamadi. AADS_REPO_ROOT set edin veya clone hedefini kontrol edin.')

REPO_ROOT = _resolve_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f'[SETUP] repo={REPO_ROOT}')


In [ ]:
# Parametreler
RUN_ALL_DATASETS = True
ADAPTER_KEY = 'tomato__leaf'

# Batch modunda PREPARED_ROOT altindaki ood/ veya oe/ iceren tum datasetler auditlenir.
PREPARED_ROOT = REPO_ROOT / 'data' / 'prepared_runtime_datasets'

# Tek dataset modunda bos birakilirsa data/prepared_runtime_datasets/<ADAPTER_KEY> kullanilir.
DATASET_ROOT = ''

# Audit artifactleri buraya yazilir.
OUTPUT_DIR = ''

# 6: konservatif near-duplicate adayi. Daha yuksek deger daha cok review satiri uretir.
NEAR_DUPLICATE_HAMMING = 6

dataset_root = Path(DATASET_ROOT or (PREPARED_ROOT / ADAPTER_KEY)).expanduser()
output_dir = Path(OUTPUT_DIR or (REPO_ROOT / '.runtime_tmp' / 'ood_oe_quality_audit')).expanduser()
if not RUN_ALL_DATASETS:
    output_dir = output_dir / ADAPTER_KEY
print(f'[PARAM] run_all={RUN_ALL_DATASETS}')
print(f'[PARAM] prepared_root={PREPARED_ROOT}')
print(f'[PARAM] dataset_root={dataset_root}')
print(f'[PARAM] output_dir={output_dir}')


In [ ]:
import json

cmd = [sys.executable, 'scripts/audit_ood_oe_quality.py']
if RUN_ALL_DATASETS:
    cmd += ['--all', '--prepared-root', str(PREPARED_ROOT)]
else:
    cmd += ['--dataset-root', str(dataset_root), '--dataset-key', ADAPTER_KEY]
cmd += [
    '--output-dir', str(output_dir),
    '--near-duplicate-hamming', str(NEAR_DUPLICATE_HAMMING),
]
completed = subprocess.run(cmd, check=False, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(completed.stdout)
if completed.returncode != 0:
    raise RuntimeError(f'Audit failed with exit code {completed.returncode}')

summary_name = 'batch_summary.json' if RUN_ALL_DATASETS else 'summary.json'
summary = json.loads((output_dir / summary_name).read_text(encoding='utf-8'))
if RUN_ALL_DATASETS:
    print('[AUDIT] batch summary CSV:', output_dir / 'batch_summary.csv')
    print('[AUDIT] per-dataset review CSVs:', output_dir / '<dataset_key>' / 'review_decisions.csv')
else:
    print('[AUDIT] review CSV:', output_dir / 'review_decisions.csv')
    print('[AUDIT] markdown report:', output_dir / 'review_report.md')

try:
    import pandas as pd
    from IPython.display import display
    preview_csv = output_dir / 'batch_summary.csv' if RUN_ALL_DATASETS else output_dir / 'review_decisions.csv'
    issues_df = pd.read_csv(preview_csv)
    display(issues_df.head(50))
except Exception as exc:
    print('[AUDIT] Pandas preview unavailable:', exc)


## Human Review

1. `review_decisions.csv` dosyasini acin.
2. Sadece emin oldugunuz satirlarda `decision` kolonunu `quarantine` yapin.
3. `target_path` hangi dosyanin tasinacagini belirler. Emin degilseniz bos birakin veya `ignore` yazin.
4. Apply hucresi dosya silmez; hedefleri `QUARANTINE_ROOT` altina tasir.


In [ ]:
APPLY_REVIEW_DECISIONS = False
# Batch modunda once REVIEW_ADAPTER_KEY secilir; her adapterin CSV'i ayridir.
REVIEW_ADAPTER_KEY = ADAPTER_KEY
active_dataset_root = PREPARED_ROOT / REVIEW_ADAPTER_KEY if RUN_ALL_DATASETS else dataset_root
DECISIONS_CSV = (output_dir / REVIEW_ADAPTER_KEY / 'review_decisions.csv') if RUN_ALL_DATASETS else (output_dir / 'review_decisions.csv')
QUARANTINE_ROOT = REPO_ROOT / '.runtime_tmp' / 'ood_oe_quality_quarantine' / REVIEW_ADAPTER_KEY

if not APPLY_REVIEW_DECISIONS:
    print('[APPLY] Kapali. Batch modunda REVIEW_ADAPTER_KEY secip ilgili review_decisions.csv duzenlendikten sonra APPLY_REVIEW_DECISIONS=True yapin.')
else:
    cmd = [
        sys.executable,
        'scripts/audit_ood_oe_quality.py',
        '--dataset-root', str(active_dataset_root),
        '--apply-decisions', str(DECISIONS_CSV),
        '--quarantine-root', str(QUARANTINE_ROOT),
    ]
    completed = subprocess.run(cmd, check=False, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f'Apply failed with exit code {completed.returncode}')


In [ ]:
# ========================================
# STEP 4: Apply quarantine decisions
# ========================================

APPLY_NOW = False  # ← SET TO True TO APPLY QUARANTINE

if not APPLY_NOW:
    print('[APPLY] ⏸ Paused. Review the decisions above, then set APPLY_NOW=True and re-run.')
    print('[APPLY] This will move flagged files to: .runtime_tmp/ood_oe_quality_quarantine/')
else:
    print('[APPLY] 🚀 Applying quarantine decisions...\n')
    
    for dataset_info in dataset_infos:
        dataset_key = dataset_info.get('dataset_key')
        dataset_root = Path(dataset_info.get('dataset_root'))
        
        if RUN_ALL_DATASETS:
            decisions_csv = OUTPUT_DIR / dataset_key / 'review_decisions.csv'
        else:
            decisions_csv = OUTPUT_DIR / 'review_decisions.csv'
        
        if not decisions_csv.exists():
            continue
        
        quarantine_root = REPO_ROOT / '.runtime_tmp' / 'ood_oe_quality_quarantine' / dataset_key
        
        # Call audit script with --apply-decisions
        cmd = [
            sys.executable,
            'scripts/audit_ood_oe_quality.py',
            '--dataset-root', str(dataset_root),
            '--apply-decisions', str(decisions_csv),
            '--quarantine-root', str(quarantine_root),
        ]
        
        result = subprocess.run(cmd, check=False, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        print(result.stdout)
        
        if result.returncode == 0:
            print(f'[APPLY] ✓ {dataset_key}: Quarantine complete')
        else:
            print(f'[APPLY] ✗ {dataset_key}: Failed (exit code {result.returncode})')

print('\n[APPLY] Done!')


## ✅ Complete

**Workflow Summary:**
1. ✓ Audit completed — found exact & near-duplicate issues
2. ✓ Issues automatically categorized (blocker vs review)
3. ✓ Decisions CSV auto-generated with quarantine decisions
4. ✓ Quarantine applied (if APPLY_NOW=True)

**Results:**
- All flagged files moved to `.runtime_tmp/ood_oe_quality_quarantine/<dataset_key>/`
- Original locations untouched (safe to retry)
- Run Cell 2 again to re-audit after cleanup


In [ ]:
# Copy audit outputs into the repo and commit them (idempotent).
from pathlib import Path
import os, shutil, subprocess, datetime

# Paths: override via env vars in Colab if needed
AUDIT_DIR = Path(os.environ.get('AUDIT_DIR', '/content/bitirmeprojesi/.runtime_tmp/ood_oe_quality_audit'))
REPO_ROOT = Path(os.environ.get('AADS_REPO_ROOT', Path.cwd()))
TARGET_REPO_DIR = REPO_ROOT / 'outputs' / 'ood_oe_quality_audit_repo'
TARGET_REPO_DIR.mkdir(parents=True, exist_ok=True)

copied = []

# Copy batch summary if present
batch_src = AUDIT_DIR / 'batch_summary.csv'
if batch_src.exists():
    dst = TARGET_REPO_DIR / 'batch_summary.csv'
    shutil.copy2(batch_src, dst)
    copied.append(dst)

# Copy per-dataset artifacts (review_decisions.csv, review_report.md, summary.json)
if AUDIT_DIR.exists():
    for d in sorted(AUDIT_DIR.iterdir()):
        if not d.is_dir():
            continue
        key = d.name
        dest_dir = TARGET_REPO_DIR / key
        dest_dir.mkdir(parents=True, exist_ok=True)
        for name in ('review_decisions.csv', 'review_report.md', 'summary.json'):
            src = d / name
            if src.exists():
                dst = dest_dir / name
                shutil.copy2(src, dst)
                copied.append(dst)

if not copied:
    print('No audit outputs found to copy from', AUDIT_DIR)
else:
    print('Copied files:')
    for p in copied:
        print(' -', p)

    # Try to commit & push only the added folder
    try:
        subprocess.run(['git', 'add', str(TARGET_REPO_DIR)], check=True)
        ts = datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z'
        msg = f'Add OOD/OE audit outputs ({ts})'
        res = subprocess.run(['git', 'commit', '-m', msg], check=False, text=True, capture_output=True)
        if res.returncode == 0:
            print('Committed audit outputs:', res.stdout.strip())
            push = subprocess.run(['git', 'push'], check=False, text=True, capture_output=True)
            print('Push stdout:', push.stdout.strip())
            print('Push stderr:', push.stderr.strip())
        else:
            out = (res.stdout or '') + (res.stderr or '')
            if 'nothing to commit' in out.lower() or 'no changes added to commit' in out.lower():
                print('No new changes to commit (files identical to repo).')
            else:
                print('Git commit failed:', out)
    except FileNotFoundError:
        print('git not found in environment; files copied locally to', TARGET_REPO_DIR)
    except Exception as e:
        print('Error during git ops:', str(e))